# Google Cloud Platform

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/gcp.ipynb)

This notebook covers the GCP services a data engineer actually uses: GCS and BigQuery as the lake and warehouse, Dataflow and Dataproc for processing, Pub/Sub for streaming, Composer for orchestration, and IAM, KMS, and VPC Service Controls for security. General infrastructure services that fill generic GCP interview lists but rarely touch data work (Cloud DNS, CDN, VPN, NAT, IoT Core, AutoML, Deployment Manager, Security Scanner, Datalab) are deliberately omitted. The *Things to remember* sections near the end compress the numbers and mistakes worth memorizing.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## The GCP mental model for data engineering

GCP's resource hierarchy — organization, folders, projects, resources — is the frame everything hangs on: IAM policies attach at any level and inherit downward, and quotas and billing are per project. Separate projects for raw, curated, and production jobs is the baseline layout.

| Layer | GCP services |
|---|---|
| Ingest | Pub/Sub (streams), Storage Transfer Service (batch), Datastream (CDC from databases) |
| Store | GCS in Standard/Nearline/Coldline/Archive classes, Hive-partitioned prefixes |
| Catalog & govern | Dataplex, Data Catalog, IAM + Conditions, BigQuery row/column policies |
| Process | Dataflow (managed Apache Beam), Dataproc (managed Spark/Hadoop), Cloud Functions (small triggers) |
| Serve | BigQuery (SQL warehouse), Bigtable/Firestore (key-value sources), Looker on top |

Cross-cutting: orchestration (Cloud Composer), observability (Cloud Monitoring and Cloud Logging), and cost control (billing export to BigQuery, budgets, slot reservations).

A common mistake is treating one project as a sandbox for raw data, curated data, and production jobs at once. Quota exhaustion, a leaked service account, or an accidental dataset delete then hits everything together. Separate projects per environment and let the hierarchy carry the policy.

## Designing a GCP data platform

The platform is a set of zones with one direction of flow. Landing holds immutable, source-aligned files under `dt=` paths. Curated holds cleaned, columnar, partitioned copies. Marts hold business logic. BigQuery can query GCS directly through external tables, but most teams load curated data natively so pruning and clustering actually work.

```sql
-- Hive-partitioned files in GCS are queryable without a load, but only _PARTITIONTIME pruning is free.
CREATE EXTERNAL TABLE curated.orders_ext
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://acme-curated-prod/orders/dt=*/*.parquet'],
  hive_partition_uri_prefix = 'gs://acme-curated-prod/orders'
);
```

The common mistake is a lake with no catalog: files under paths that only one engineer remembers. Register external tables or Dataplex entries from the job that writes the files, not months later when someone asks what a prefix contains.

## Choosing the engine: Dataflow, Dataproc, BigQuery, or Composer

Choose by the shape of the work, not by brand. Dataflow is the default for streaming and for Beam pipelines that must run the same code in batch. Dataproc is the path for existing Spark or Hadoop code and for libraries that Beam does not wrap. BigQuery is the default when the data already lives in the warehouse and the transform is SQL. Composer is the control plane that starts the others and handles cross-system dependencies.

A second axis is cost structure: BigQuery bills per byte scanned or per slot-hour, Dataflow bills per worker-hour plus shuffle, Dataproc bills per cluster-minute including idle, and Composer bills per environment-hour even when no DAG runs. A cheap engine left running is often more expensive than a pricey engine that stops.

The common mistake is running Dataproc for a transform whose inputs and outputs are both BigQuery tables. That reads bytes out of the warehouse, shuffles them on VMs, and writes them back. Push the SQL to BigQuery unless Spark libraries are genuinely required.

| Need | Choose | Why | Watch out |
|---|---|---|---|
| Streaming ETL, autoscaling, Beam | Dataflow | serverless, exactly-once with BigQuery sink | worker sizing and shuffle costs |
| Existing Spark/Hadoop, custom libs | Dataproc | full cluster control, GPUs, Hive metastore | idle clusters burn money |
| SQL modeling and BI | BigQuery | serverless, slot-based pricing | partition pruning is on you |
| Cross-system scheduling | Cloud Composer | Airflow DAGs, sensors, retries | environment cost ~$300+/mo |

When NOT to use Dataflow: a small, hourly SQL transform belongs in BigQuery scheduled queries, and a one-off file conversion belongs in a Cloud Run job — a Beam pipeline you do not need is operational overhead.

## Why GCP for data engineering: honest trade-offs

Three advantages are structural rather than marketing. Storage and compute separate, so a warehouse can hold years of data without an always-on cluster. Autoscaling is native to BigQuery slots and Dataflow workers, so capacity follows the job. Global networking and IAM inheritance mean one policy spans regions.

The honest trade-offs: byte-scanned pricing rewards discipline because a missing partition filter is a cost event, the managed services are not portable to other clouds, and the sheer number of services makes governance a design problem rather than an afterthought.

The common mistake in an interview is to list advantages with no matching cost. Pair every benefit with the operational habit it requires: serverless needs filter discipline, autoscaling needs a `--max-workers` cap.

## Storage services by access pattern

Pick a storage service by access pattern and consistency requirement, not by capacity. Object storage for files and backups, relational for transactions, wide-column for high-throughput key lookups, document for application state, and a warehouse for analytical scans. The same bytes often live in two of these by design, one for serving and one for analytics.

```bash
# Storage classes are a lifecycle decision, not a one-time setting
gcloud storage buckets create gs://acme-raw-prod --location=US --default-storage-class=STANDARD
gcloud storage buckets update gs://acme-raw-prod --lifecycle-file=lifecycle.json
cat lifecycle.json
{
  "rule": [
    {"action": {"type": "SetStorageClass", "storageClass": "NEARLINE"}, "condition": {"age": 30}},
    {"action": {"type": "SetStorageClass", "storageClass": "COLDLINE"}, "condition": {"age": 120}},
    {"action": {"type": "Delete"}, "condition": {"age": 1095}}
  ]
}
```

A common mistake is to size the choice by volume alone: a 10 TB archive that is read weekly belongs in Nearline or Standard, not Archive, because retrieval cost and minimum storage duration can exceed the per-GB saving. Estimate reads, not gigabytes.

## BigQuery: how it actually works

BigQuery is one platform with four pieces. Colossus stores the columnar data durably. Dremel executes SQL as a tree of stages where leaves scan and intermediate stages shuffle. Borg allocates the slots that run those stages. The streaming buffer holds freshly inserted rows before they are compacted into managed storage, which is why a just-inserted row can be invisible to a `SELECT *` over an old snapshot.

```sql
-- Slots and shuffle are the two execution numbers worth knowing
SELECT job_id, ROUND(total_bytes_billed / POW(1024, 4), 3) AS tib_billed,
       ROUND(total_slot_ms / 3600000.0, 2) AS slot_hours,
       ROUND(shuffle_output_bytes / POW(1024, 3), 2) AS gib_shuffled
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
  AND statement_type = 'SELECT'
ORDER BY total_slot_ms DESC LIMIT 10;
```

The common mistake is to treat BigQuery as a database you can query any way you like. There are no indexes and no tuning knobs on the storage engine, so partitioning, clustering, and the shape of the query are the only levers. An unpruned `SELECT *` reads the whole table every time.

```sql
-- Partitioning and clustering are the two biggest cost levers
CREATE TABLE curated.orders (
  order_id     STRING NOT NULL,
  customer_id  STRING,
  order_ts     TIMESTAMP,
  country      STRING,
  amount       NUMERIC
)
PARTITION BY DATE(order_ts)
CLUSTER BY country, customer_id
OPTIONS (partition_expiration_days = 1095,
         require_partition_filter = TRUE);
```

```bash
# Always dry-run before a large query: it reports bytes, costs nothing
bq query --dry_run --use_legacy_sql=false \
  'SELECT country, SUM(amount) FROM `acme-data-prod.curated.orders`
   WHERE DATE(order_ts) = "2026-10-04" GROUP BY country'
```

`require_partition_filter = TRUE` turns an accidental full-table scan into an error rather than a $500 surprise. Trade-off: appending to or replacing a single partition is a per-partition operation (`WRITE_APPEND` or `WRITE_TRUNCATE` against a `table$YYYYMMDD` decorator), and over-clustering on high-cardinality columns increases metadata overhead without helping pruning.

## Moving data into GCP: transfer, Datastream, and loads

Batch file movement and database replication are different problems. Storage Transfer Service copies objects from S3, HTTP lists, or on-prem agents into a bucket and can run on a schedule. BigQuery Data Transfer Service pulls from supported SaaS and database sources directly into datasets. Datastream reads the database log for change data capture so inserts, updates, and deletes all arrive. Pub/Sub plus Dataflow or the Storage Write API covers events.

```sql
-- Datastream writes to GCS first; this external table is the landing contract
CREATE EXTERNAL TABLE landing.orders_cdc
OPTIONS (
  format = 'AVRO',
  uris = ['gs://acme-raw-prod/datastream/orders/*.avro'],
  hive_partition_uri_prefix = 'gs://acme-raw-prod/datastream/orders'
);
```

The common mistake is choosing a one-shot transfer for a source that keeps changing. If the source is a database, the answer is CDC or a scheduled incremental extract with a watermark column, never a full copy on a cron that silently drifts.

## Dataflow and Apache Beam

Apache Beam is the programming model and Dataflow is one runner for it. A pipeline is a graph of `PCollection`s flowing through `PTransform`s, and the runner decides how to cut that graph into stages and execute them. Transform fusion merges adjacent cheap steps into one stage, which is why a chain of `Map`s costs no extra shuffle. Batch sources bound the input; streaming sources are unbounded and need windows and triggers to produce results.

```bash
# The same pipeline file runs on the local DirectRunner and then on Dataflow
python orders_pipeline.py --runner=DirectRunner --input=./sample.jsonl
python orders_pipeline.py --runner=DataflowRunner --project=acme-data-prod \
  --region=us-central1 --temp_location=gs://acme-temp/dataflow \
  --staging_location=gs://acme-temp/staging --max_num_workers=20
```

The common mistake is testing only on DirectRunner. DirectRunner executes nearly everything in memory and hides coder, serialization, and determinism problems that Dataflow rejects at submission. Run a small job on Dataflow before the real one.

Two managed backends change Dataflow economics. The Streaming Engine moves windowing state and timers out of the worker JVM to a service, so workers can be smaller and the pipeline can update without draining state. The Dataflow Shuffler moves shuffle data off worker persistent disks to a service, so you no longer size disks for the largest shuffle. Together they let a streaming job scale workers up and down without losing state.

```yaml
# Pipeline options for a streaming job: engine on, workers capped
experiments:
  - use_runner_v2
  - enable_streaming_auto_sharding
streaming: true
enable_streaming_engine: true
num_workers: 2
max_num_workers: 20
autoscaling_algorithm: THROUGHPUT_BASED
worker_machine_type: n2-standard-4
disk_size_gb: 100
```

The common mistake is to leave `--max-workers` unset on a streaming job and blame the bill on traffic. Autoscaling without a cap will follow a bad source or a hot key all the way to the quota. Set the cap, the disk size, and an alert on system lag.

```python
# Streaming windowed aggregation with a watermark and late-data handling (Apache Beam on Dataflow)
import json

import apache_beam as beam
from apache_beam.transforms.trigger import AccumulationMode, AfterProcessingTime, AfterWatermark
from apache_beam.transforms.window import FixedWindows

with beam.Pipeline() as p:
    (
        p
        | beam.io.ReadFromPubSub(topic="projects/acme/topics/orders")
        | beam.Map(lambda b: json.loads(b.decode("utf-8")))
        | beam.Map(lambda e: (e["country"], e["amount"]))
        | beam.WindowInto(
            FixedWindows(60),
            trigger=AfterWatermark(early=AfterProcessingTime(30), late=AfterProcessingTime(60)),
            allowed_lateness=300,
            accumulation_mode=AccumulationMode.ACCUMULATING,
        )
        | beam.CombinePerKey(sum)
        | beam.io.WriteToBigQuery("acme:curated.revenue_1m", schema="country:STRING,revenue:FLOAT")
    )
```

The cell below simulates what that pipeline does with event time, the watermark, and allowed lateness, in plain Python:

In [ ]:
from collections import defaultdict

WINDOW_S, ALLOWED_LATENESS_S = 60, 300

# (event_time_s, arrival_order, country, amount): events arrive out of order.
events = [(5, 1, "DE", 10.0), (61, 2, "DE", 5.0), (30, 3, "FR", 7.0), (130, 4, "DE", 1.0),
          (20, 5, "DE", 2.0),      # late but within allowed lateness: updates window [0, 60)
          (500, 6, "FR", 3.0),     # advances the watermark far ahead
          (10, 7, "DE", 99.0)]     # later than watermark - allowed lateness: dropped

panes = defaultdict(float)
watermark, dropped = 0, []
for event_time, _, country, amount in sorted(events, key=lambda e: e[1]):  # process in arrival order
    window_start = event_time // WINDOW_S * WINDOW_S
    if window_start + WINDOW_S + ALLOWED_LATENESS_S <= watermark:
        dropped.append((event_time, country, amount))
        continue
    panes[(window_start, country)] += amount  # ACCUMULATING: each firing includes earlier data
    watermark = max(watermark, event_time)     # a perfect watermark: max event time seen

for (start, country), total in sorted(panes.items()):
    print(f"[{start:>3}, {start + WINDOW_S:>3}) {country}: {total}")
print("dropped as too late:", dropped)
assert panes[(0, "DE")] == 12.0 and dropped == [(10, "DE", 99.0)]

Sizing rule of thumb: one worker per 1-2 vCPUs of steady load, plus headroom; use `--max-workers` to cap cost and `--num-workers` to control cold start. Streaming jobs with the Streaming Engine off will pay for shuffle on persistent disks.

## Cloud Dataproc: managed Spark and Hadoop

Dataproc is a control plane that creates a cluster on Compute Engine and submits jobs to it. An ephemeral cluster is the default production pattern: create, run, delete, with `--max-idle` as a safety net. Autoscaling policies add and remove workers from YARN metrics, and secondary workers can be Spot VMs to absorb shuffle-heavy stages cheaply.

```bash
# Autoscaling policy used by the ephemeral cluster
gcloud dataproc autoscaling-policies import orders-etl --region=us-central1 \
  --source=autoscaling.yaml
gcloud dataproc clusters create etl-tmp --region=us-central1 \
  --autoscaling-policy=orders-etl --num-workers=4 \
  --secondary-worker-type=spot --num-secondary-workers=20 \
  --optional-components=JUPYTER,HIVE_WAREHOUSE --max-idle=30m
```

The common mistake is leaving the cluster up because the next job is tomorrow. Cluster idle time is billed like work, and a forgotten development cluster with GPUs can cost more than the whole pipeline. Prefer ephemeral clusters or Dataproc Serverless.

## Security layers: IAM, VPC Service Controls, and perimeters

Security is layered because no single control is sufficient. IAM answers who can call which API. VPC Service Controls draws a perimeter so data cannot be copied out even by a principal that holds the permission. Cloud KMS gives you the key that encrypts the bytes. Audit logs record who did what. Each layer fails independently, so an interview answer should name all four.

```bash
# Perimeter around the project: even an owner cannot exfiltrate the datasets
# without an access level that matches the request context
gcloud access-context-manager perimeters create acme_data_perimeter \
  --policy=123456789 --title="acme data" --resources=projects/987654321 \
  --restricted-services=bigquery.googleapis.com,storage.googleapis.com \
  --access-levels=acme_corp_access
```

The common mistake is VPC Service Controls with no access level, which locks out the legitimate analysts along with the attacker. Build the access level first, test in dry-run mode, then enforce.

The IAM policy below grants least privilege: read one curated dataset, run Dataflow, and no project-owner role.

```json
{
  "bindings": [
    {
      "role": "roles/bigquery.dataViewer",
      "members": ["serviceAccount:dataflow-runner@acme-data-prod.iam.gserviceaccount.com"],
      "condition": {
        "title": "curated-dataset-only",
        "expression": "resource.name.startsWith('projects/acme-data-prod/datasets/curated')"
      }
    },
    {
      "role": "roles/dataflow.worker",
      "members": ["serviceAccount:dataflow-runner@acme-data-prod.iam.gserviceaccount.com"]
    }
  ]
}
```

## Pub/Sub: topics, subscriptions, and cursors

A topic is a named stream; a subscription is a durable cursor over it. Nothing is delivered unless a subscription exists, and each subscription has its own ack state, retention, and retry policy. An ordering key makes messages with the same key deliver in publish order to one subscriber, and exactly-once delivery means a redelivery after a successful ack is suppressed. The ack deadline is a lease: if the client does not ack or extend it, the message is redelivered.

```bash
# Subscription with ordering, a retry policy, and a dead-letter topic
gcloud pubsub subscriptions create orders-v1-sub \
  --topic=orders-v1 --ack-deadline=60 --message-retention-duration=7d \
  --enable-message-ordering --enable-exactly-once-delivery \
  --dead-letter-topic=orders-v1-deadletter --max-delivery-attempts=5 \
  --min-retry-delay=10s --max-retry-delay=600s
```

The common mistake is ordering keys with an unbounded number of keys or with concurrent consumers, which serializes throughput and can still reorder across keys. The second mistake is a dead-letter topic whose subscription does not grant the Pub/Sub service account publish rights, so failures are dropped instead of parked.

## Cloud Storage: buckets, objects, and throughput

A bucket is a globally unique namespace bound to a location; an object is an immutable byte string plus metadata, addressed by name. Updating an object is a rewrite that creates a new generation, which is why versioning is cheap insurance. Parallel composite uploads split a large file into parts and compose them, producing an object whose `crc32c` is valid but whose `md5` is absent.

```bash
# Throughput scales with parallel workers, not with file size
gcloud storage cp --recursive ./exports/orders/ gs://acme-raw-prod/orders/dt=2026-10-04/
gcloud storage rsync --recursive --delete-unmatched-destination-objects \
  ./exports/ gs://acme-raw-prod/exports/
gcloud storage hash gs://acme-raw-prod/orders/dt=2026-10-04/part-00000.parquet
```

Request pricing is the gotcha. Class A operations (writes, lists, copies) cost several times a Class B read, and every object touched by a list or a scan counts. A prefix with a million tiny files costs far more to list and read than a few hundred compacted files, even at identical total bytes.

## Cloud Composer: managed Airflow

Composer is Airflow running on a GKE cluster that Google operates, with Cloud SQL for the metadata database and a GCS bucket synced to the workers' `dags/` folder. You write DAGs; the environment handles the scheduler, the web server, the workers, and the database. Environment size sets the floor on workers and the scheduler capacity, and worker autoscaling adds task slots under load.

```bash
# DAGs deploy by copying to the environment's dags prefix
gcloud composer environments describe acme-airflow --location=us-central1 \
  --format='value(config.dagGcsPrefix)'
gcloud storage cp orders_hourly.py gs://us-central1-acme-airflow-abc123-bucket/dags/
```

The common mistake is a DAG that does real work at import time. Airflow parses every DAG file continuously on the scheduler, so a top-level API call or a heavy query slows the whole environment for every DAG. Keep the file declarative and put work in operators.

```python
# DAG: GCS -> Dataflow template -> BigQuery check
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.google.cloud.operators.dataflow import DataflowStartFlexTemplateOperator
from airflow.providers.google.cloud.sensors.gcs import GCSObjectExistenceSensor

with DAG(
    dag_id="orders_hourly",
    schedule="0 * * * *",
    start_date=datetime(2026, 1, 1),
    catchup=False,
    default_args={"retries": 2, "retry_delay": timedelta(minutes=5)},
) as dag:
    wait_for_file = GCSObjectExistenceSensor(
        task_id="wait_for_file",
        bucket="acme-raw-prod",
        object='orders/dt={{ ds }}/hour={{ data_interval_start.hour }}/_SUCCESS',
        timeout=3600,
        poke_interval=60,
    )
    run_dataflow = DataflowStartFlexTemplateOperator(
        task_id="run_dataflow",
        body={
            "launchParameter": {
                "jobName": "orders-hourly",
                "containerSpecGcsPath": "gs://acme-code/templates/orders.json",
                "parameters": {"input": "gs://acme-raw-prod/orders/", "output": "curated.orders"},
            }
        },
        location="us-central1",
    )
    wait_for_file >> run_dataflow
```

Trade-off: Composer is the right tool for cross-system dependencies and sensors, but it costs money even when idle and is a poor fit for per-minute latency — use Dataflow triggers or Cloud Scheduler for those.

## Monitoring and troubleshooting data pipelines

Observability has three signals: metrics for rates and thresholds, logs for events, and traces for latency across services. Cloud Monitoring stores metrics and evaluates alerting policies, Cloud Logging stores and queries log entries and can turn a filter into a counter metric, and Cloud Trace follows a request across hops.

```bash
# Read recent errors for a Dataflow job
gcloud logging read 'resource.type="dataflow_step" severity>=ERROR' \
  --limit=20 --format='table(timestamp, resource.labels.job_name, textPayload)'
gcloud logging metrics create orders_dlq_errors \
  --description="Dead-letter records" \
  --log-filter='resource.type="dataflow_step" jsonPayload.stage="dlq"'
```

The common mistake is alerting on causes rather than symptoms. Worker CPU above 80 percent is normal during a healthy burst; what users feel is data freshness, backlog age, and job failure. Alert on those, and use the cause metrics to debug after the page.

Also monitor BigQuery slots, Pub/Sub oldest unacked message age, GCS request latency, and Composer task duration. The trick is to alert on user-visible symptoms (freshness, backlog age) rather than raw resource metrics.

Triage is a loop: find the symptom, find the component that owns it, read that component's own metrics, change one thing, and compare the same numbers. BigQuery owns query latency and bytes scanned. Dataflow owns system lag and worker CPU. Pub/Sub owns oldest unacked message age. Dataproc owns YARN pending memory and shuffle spill. Composer owns scheduler heartbeat and task queue time.

```bash
# Dataflow job metrics that separate slow sources from slow workers
gcloud dataflow jobs list --region=us-central1 --status=active \
  --format='table(id, name, currentStateTime)'
gcloud logging read 'resource.type="dataflow_step" severity>=WARNING' --limit=20
```

The common mistake is to guess at the cause and change three things at once. With three changes you learn nothing about which one helped. Change one lever, keep the measurement window identical, and record the before and after numbers.

## High availability and disaster recovery

A zone is one failure domain, a region is a set of zones, and multi-region is a set of regions. HA means surviving a zone failure without manual action; DR means surviving a region failure, usually with a documented rebuild or a warm copy. Google publishes SLAs per service, and the monthly uptime target is the promise you design against.

```bash
# Regional Cloud SQL gives a synchronous standby in a second zone
gcloud sql instances create acme-pg --database-version=POSTGRES_16 --region=us-central1 \
  --availability-type=REGIONAL --tier=db-custom-4-16384 --backup --enable-point-in-time-recovery
```

The common mistake is to equate redundancy with backup. A dual-region bucket survives a region outage but happily replicates a bad delete or a ransomware overwrite to both regions. Keep versioning or an export in a separate project for the recovery case.

Disaster recovery starts from two numbers: recovery point objective, how much data you can lose, and recovery time objective, how long you can be down. Those numbers select a pattern. Backup and restore is cheapest and slowest. Pilot light keeps data replicated and infrastructure as code. Warm standby runs a scaled-down copy. Active-active multi-region is the most expensive and the only one with near-zero RTO.

```bash
# Copy a critical dataset to a DR region on a schedule
gcloud transfer jobs create --source-agent-pool=acme-pool \
  gs://acme-critical-prod/ gs://acme-dr-prod/critical/ --schedule-repeats-every=1d
bq mk --transfer_config --data_source=cross_region_copy --project_id=acme-data-prod \
  --target_dataset=curated_dr --display_name='curated nightly DR copy' \
  --params='{"source_dataset_id":"curated","source_project_id":"acme-data-prod","destination_dataset_id":"curated_dr"}'
```

The common mistake is a DR plan that has never been executed. Restore one table and one instance into a clean project every quarter and record the wall-clock time; the measured RTO is usually several times the estimate.

## Cloud SQL as a managed relational source

Cloud SQL is a managed VM running MySQL, PostgreSQL, or SQL Server with automated backups, patching, and optional replication. A regional instance keeps a synchronous standby in another zone for failover; a read replica is asynchronous and can live in another region. Point-in-time recovery replays the binary log onto a new instance from a chosen timestamp.

```bash
# Create, then connect through the connector rather than a public IP
gcloud sql instances create acme-orders --database-version=POSTGRES_16 \
  --region=us-central1 --tier=db-custom-2-8192 --storage-auto-increase --no-assign-ip
gcloud sql databases create orders --instance=acme-orders
```

The common mistake is pointing a BI tool at the production instance. Dashboards hold long transactions and connections that starve the OLTP workload; replicate into BigQuery and let the warehouse absorb the scan.

## Spanner and Firestore: the other database sources

Spanner is a globally distributed relational database built on TrueTime and Paxos. Data is split into ranges, each range has a replica set, and writes go through a leader elected per split. Reads at the current timestamp are strongly consistent because the commit timestamp comes from synchronized clocks plus a wait-out uncertainty interval. Interleaved tables physically co-locate child rows with their parent to keep joins local.

```bash
# Capacity is bought in processing units; 1000 units is one node
gcloud spanner instances create acme-oltp --config=regional-us-central1 \
  --processing-units=1000 --description="Global orders"
gcloud spanner databases execute-sql orders --instance=acme-oltp \
  --sql='SELECT COUNT(*) FROM Orders'
```

The common mistake is choosing Spanner for analytics because it scales horizontally. Every analytical scan crosses splits on a system tuned for small transactions, and the bill reflects provisioned capacity that idles overnight. Warehouse the data instead.

Firestore in Datastore mode is a schemaless document store with automatic indexes and strong consistency inside an ancestor group. Queries that cross entity groups are eventually consistent unless you use an ancestor query. Composite indexes are declared in `index.yaml`, and a query that needs an index it does not have fails at runtime rather than scanning.

```python
# Entities and a strongly consistent ancestor query
from google.cloud import datastore

client = datastore.Client(project="acme-data-prod")
key = client.key("Order", "order-1001")
client.put(datastore.Entity(key=key, order_total=42.5))
customer_key = client.key("Customer", "cust-7")
query = client.query(kind="Order", ancestor=customer_key).order("-order_total")
rows = [(e.key.name, e["order_total"]) for e in query.fetch(limit=5)]
assert isinstance(rows, list)
```

The common mistake is a monotonically increasing key such as a timestamp, which concentrates writes on one range and creates a hot spot. Use a distributed key or let Firestore generate one.

## Cloud IAM for data platforms

IAM is an allow-only model. Policies are additive across the hierarchy, so a grant cannot be taken away by a narrower one lower down; it can only be denied by an explicit deny policy, which is a separate feature. A role is a bundle of permissions, a member is an identity, and a binding joins them at a resource. Conditions narrow a binding by resource name, request time, or request attributes.

```bash
# Minimum to run a BigQuery job, then read exactly one dataset
gcloud projects add-iam-policy-binding acme-data-prod \
  --member="serviceAccount:dataflow-runner@acme-data-prod.iam.gserviceaccount.com" \
  --role="roles/bigquery.jobUser"
bq add-iam-policy-binding \
  --member="serviceAccount:dataflow-runner@acme-data-prod.iam.gserviceaccount.com" \
  --role="roles/bigquery.dataViewer" acme-data-prod:curated
```

The common mistake is to grant `roles/owner` or `roles/editor` to a service account to unblock a task and never revisit it. Primitive roles span every service in the project. Grant the narrowest predefined role, then downscope to the dataset or table.

## Encryption at rest: Google-managed, CMEK, and CSEK

Every byte in GCP is encrypted at rest with AES-256 using keys Google manages and rotates. CMEK replaces the key-encryption key with one you own in Cloud KMS, which gives you the ability to revoke access and to see key usage in audit logs. CSEK and customer-provided keys go further and make availability your problem. In transit everything is TLS, and inside Google's network it is ALTS.

```bash
# One key ring, one key, used as the default key for a BigQuery dataset
gcloud kms keyrings create acme-bq --location=us-central1
gcloud kms keys create curated --location=us-central1 --keyring=acme-bq \
  --purpose=encryption --rotation-period=90d --next-rotation-time=2027-01-01T00:00:00Z
bq mk --dataset \
  --default_kms_key=projects/acme-data-prod/locations/us-central1/keyRings/acme-bq/cryptoKeys/curated \
  acme-data-prod:curated_encrypted
```

The common mistake is to assume CMEK adds protection against an attacker who already has the service account. CMEK protects against Google-side access and gives you a kill switch; IAM still decides who reads the plaintext.

## Monitoring and analyzing cost

Cost visibility comes from three instruments. Billing export writes line items to BigQuery so you can join spend to labels and usage. Budgets alert on thresholds and can trigger Pub/Sub for automation. Labels and project boundaries are the allocation mechanism: a resource with no label cannot be charged back to a team.

```sql
-- Monthly spend by service and label from the billing export
SELECT invoice.month AS month, service.description AS service,
       ROUND(SUM(cost), 2) AS usd
FROM `acme-billing.billing_export.gcp_billing_export_v1_0123AB`
WHERE invoice.month >= '202609'
GROUP BY month, service
ORDER BY usd DESC LIMIT 20;
```

The common mistake is to optimize by looking at the biggest service rather than the biggest driver. BigQuery storage is often large but flat, while a single Dataflow job left at 50 workers overnight is small in the list and large in the delta. Compare month over month per line item.

```sql
-- BigQuery cost by label/team from the audit logs / INFORMATION_SCHEMA
SELECT
  REGEXP_EXTRACT(job_id, r'^([^.]+)') AS team,
  ROUND(SUM(total_bytes_billed) / POW(1024, 4), 2) AS tb_billed,
  ROUND(SUM(total_bytes_billed) / POW(1024, 4) * 6.25, 2) AS usd,
  COUNT(*) AS queries
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_TRUNC(CURRENT_TIMESTAMP(), DAY)
GROUP BY team
ORDER BY usd DESC
LIMIT 20;
```

The biggest BigQuery levers: partition filters, clustering, `SELECT` only needed columns, materialized views for repeated aggregations, and switching stable heavy dashboards to flat-rate slots with reservations.

## Optimizing ingestion throughput

Ingestion throughput is decided before the data reaches the warehouse. Batch messages into fewer, larger requests, land files at 128-512 MB, compress with Snappy or ZSTD, and partition the path by the columns that queries filter. Pub/Sub flow control and Dataflow autoscaling provide backpressure so a slow sink does not force the source to buffer without limit.

```python
# Pub/Sub publisher batching: fewer, larger requests, with flow control on the subscriber
from google.cloud import pubsub_v1

batch = pubsub_v1.types.BatchSettings(max_messages=1000, max_bytes=1024 * 1024, max_latency=0.05)
publisher = pubsub_v1.PublisherClient(batch_settings=batch)
subscriber = pubsub_v1.SubscriberClient()
flow = pubsub_v1.types.FlowControl(max_messages=1000, max_bytes=100 * 1024 * 1024)
assert batch.max_messages == 1000 and flow.max_messages == 1000
```

The common mistake is to treat the transport guarantee as a deduplication guarantee. At-least-once delivery means your sink must be idempotent on a business key, because a message can arrive twice after any retry.

## Archiving and long-term storage

Each storage class has a minimum storage duration: Nearline 30 days, Coldline 90 days, Archive 365 days. Deleting or rewriting an object before that duration is met bills the remaining time as if it were stored. Retrieval also differs: Nearline and Coldline have per-GB retrieval fees, Archive has a longer latency and a higher retrieval cost. Autoclass moves objects between classes automatically based on access.

```bash
# Object Lifecycle Management rule set: transition then expire
gcloud storage buckets update gs://acme-raw-prod --lifecycle-file=lifecycle.json
cat lifecycle.json
{
  "rule": [
    {"action": {"type": "SetStorageClass", "storageClass": "NEARLINE"}, "condition": {"age": 30, "matchesPrefix": ["orders/"]}},
    {"action": {"type": "SetStorageClass", "storageClass": "COLDLINE"}, "condition": {"age": 120, "matchesPrefix": ["orders/"]}},
    {"action": {"type": "SetStorageClass", "storageClass": "ARCHIVE"}, "condition": {"age": 365, "matchesPrefix": ["orders/"]}},
    {"action": {"type": "Delete"}, "condition": {"age": 2555, "matchesPrefix": ["orders/"]}}
  ]
}
```

The common mistake is a lifecycle rule that transitions data a pipeline still reads every day. Each read of an Archive object pays a retrieval fee and adds latency, so the saving disappears. Check access logs before you transition.

## Access control beyond IAM: row and column policies

Access control has four layers: who, on what, under which conditions, and with which row or column filter. IAM covers who and on what. Conditions narrow a binding by resource name or request context. Row access policies and column policy tags apply filters inside the table so a query cannot see rows it should not. Authorized views expose a filtered projection without granting the base table.

```sql
-- Row-level and column-level control inside one table
CREATE ROW ACCESS POLICY region_filter ON curated.orders
GRANT TO ('group:analysts-eu@acme.com')
FILTER USING (country IN ('DE', 'FR', 'ES'));
ALTER TABLE curated.customers
ALTER COLUMN email SET OPTIONS (
  policy_tags = 'projects/acme-data-prod/locations/us/taxonomies/1/policyTags/2');
```

The common mistake is to grant `bigquery.dataViewer` on the dataset and then rely on the dashboard to filter rows. Anyone with the dataset grant can query the base table directly. Put the filter in a row access policy or in an authorized view.

## Optimizing BigQuery query processing

Optimization is mostly about moving less data and doing less per row. Prune scans with partition filters and clustering. Read only the columns you need. Aggregate before joining when the join multiplies rows. Use combiners instead of collecting lists. Right-size the workers so that a shuffle has enough parallelism but not more than the data supports.

```sql
-- Pre-aggregate the large side before the join so the shuffle carries fewer rows
WITH orders AS (
  SELECT customer_id, SUM(amount) AS revenue
  FROM curated.orders WHERE order_ts >= TIMESTAMP '2026-10-01'
  GROUP BY customer_id
)
SELECT c.region, SUM(o.revenue) AS revenue
FROM orders AS o JOIN curated.customer AS c USING (customer_id)
GROUP BY c.region;
```

The common mistake is to add workers to a skewed job. If one key carries most of the rows, one task does most of the work no matter how many workers exist. Fix the key or salting strategy first.

## Replication and synchronization across regions

Replication is described by direction, latency, and failure domain. Synchronous replication commits on both sides before acknowledging, so RPO is zero and latency pays for it. Asynchronous replication acknowledges locally and catches up later, so RPO is the lag. Datastream replicates database changes into GCS or BigQuery, and Storage Transfer Service replicates objects between buckets.

```bash
# Cross-region read replica for analytics, plus a dual-region bucket
gcloud sql instances create acme-pg-replica --master-instance-name=acme-pg \
  --region=us-east1 --tier=db-custom-4-16384
gcloud storage buckets create gs://acme-critical-prod \
  --location=us-central1 --placement=us-east1 --rpo=ASYNC_TURBO
```

The common mistake is to point analytics at an asynchronous read replica and assume the numbers are complete. Check the replica lag metric in the query path, or replicate into the warehouse and query a snapshot with a known watermark.

## Cloud Functions for event-driven tasks

Cloud Functions 2nd gen is a wrapper around Cloud Run, so it shares the same concurrency model, the same container runtime, and a longer timeout. Eventarc delivers events with at-least-once semantics, which means a function can run twice for one event. That is why the handler must be idempotent on the event id or on the object generation.

The common mistake is to write a function that writes a file into the same bucket it watches. That triggers itself again and can loop until the budget stops it. Write outputs to a different prefix or bucket.

```python
# validate_orders/main.py: reject a malformed file before it reaches the warehouse
import functions_framework
from google.cloud import storage


@functions_framework.cloud_event
def validate(cloud_event):
    data = cloud_event.data
    bucket, name = data["bucket"], data["name"]
    if not name.endswith(".parquet"):
        raise ValueError(f"unexpected object {name}")
    blob = storage.Client().bucket(bucket).blob(name)
    print(f"validated gs://{bucket}/{name} size={blob.size} crc32c={blob.crc32c}")
```

## Tuning GCS layout for BigQuery and Dataflow

File layout is the first performance decision because both BigQuery and Dataflow pay per file, not only per byte. Target 128-512 MB per Parquet or Avro file after compression. BigQuery computes schema and statistics per file, so ten thousand 10 MB files cost ten thousand metadata operations and prune poorly. Dataflow parallelizes reads by splitting each file, so a single huge file still reads in parallel, but too few files starves the worker pool.

Naming matters for two reasons. A Hive-style path such as `orders/dt=2026-10-04/hour=07/` is what `hive_partition_uri_prefix` turns into partitions, and a random or hash prefix avoids the GCS hot-spot that sequential names can create when many writers append at high rate. Write to a temporary prefix and rename or copy into the final prefix once the file is complete, so a reader never sees a partial object. An `_SUCCESS` marker is the signal a downstream sensor waits on.

```bash
# Compact many small files into a few large ones, then keep the layout tidy
gcloud storage ls --recursive gs://acme-raw-prod/orders/dt=2026-10-04/ | wc -l
gcloud storage cp --recursive gs://acme-raw-prod/orders/dt=2026-10-04/ \
  gs://acme-curated-prod/orders/dt=2026-10-04/
gcloud storage rm --recursive gs://acme-raw-prod/orders/dt=2026-10-04/part-00*.json
# Marker the pipeline's sensor waits on
gcloud storage cp /dev/null gs://acme-curated-prod/orders/dt=2026-10-04/_SUCCESS
```

```python
# Beam compaction sketch: read small files, write ~256 MB shards
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

options = PipelineOptions(["--runner=DataflowRunner", "--region=us-central1"],
                          temp_location="gs://acme-temp/dataflow")
with beam.Pipeline(options=options) as p:
    (
        p
        | "Read" >> beam.io.ReadFromText("gs://acme-raw-prod/orders/dt=2026-10-04/*.jsonl")
        | "Write" >> beam.io.WriteToParquet(
            "gs://acme-curated-prod/orders/dt=2026-10-04/part",
            schema="order_id:STRING,customer_id:STRING,amount:FLOAT",
            file_name_suffix=".parquet",
        )
    )
```

```sql
-- An external table over the compacted layout prunes on the Hive partition columns
CREATE EXTERNAL TABLE curated.orders_ext
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://acme-curated-prod/orders/dt=*/*.parquet'],
  hive_partition_uri_prefix = 'gs://acme-curated-prod/orders'
);
SELECT dt, SUM(amount) FROM curated.orders_ext
WHERE dt BETWEEN '2026-10-01' AND '2026-10-07' GROUP BY dt;
```

Worked example: a nightly job lands 240 GB of JSON as 480,000 files of 512 KB. The BigQuery external table scan takes 22 minutes and lists one million objects to find them. A compaction job rewrites the day into 960 Parquet files of 256 MB. The same scan then reads 240 GB in under three minutes, and the request count drops by three orders of magnitude. The bill improvement comes from Class A operations and latency, not from storage size.

The common mistake is to optimize bytes and ignore object count. Storage cost is per GB, but list, read, and metadata costs are per object, so a prefix of tiny files is the expensive shape even when it is small. The second mistake is a lifecycle rule that deletes the temporary prefix before the compaction job has copied from it.

In [ ]:
# Decide whether a prefix needs compaction: objects per GB is the signal
total_gb = 240.0
objects = 480_000
average_mb = total_gb * 1024 / objects
needs_compaction = average_mb < 128
assert round(average_mb, 2) == 0.51
assert needs_compaction is True
target_files = int(total_gb * 1024 / 256)
print(f"average file {average_mb:.2f} MB -> compact to about {target_files} files")

## Sizing Dataflow workers and autoscaling

Sizing starts from throughput, not from a machine type. Measure the per-vCPU throughput of the heaviest transform, multiply by the vCPUs on the candidate worker, and divide the steady input rate by that number. Add 30 percent headroom for skew and backlog drain. Then set `num_workers` to the cold-start floor and `max_num_workers` to the cost ceiling. Autoscaling grows between them from the backlog, and `--max_num_workers` is what stops a bad source from spending without limit.

The Streaming Engine and the Dataflow Shuffler change the sizing inputs. With the streaming engine, windowing state and timers live in a service, so worker heap is not sized for state and a pipeline can update without draining. With the shuffler, shuffle data does not consume worker persistent disk, so `disk_size_gb` is not sized for the largest shuffle. `enable_streaming_auto_sharding` splits hot keys at runtime, which helps when one key carries a disproportionate share.

```yaml
# Streaming pipeline options: cap cost, keep state and shuffle off the workers
streaming: true
enable_streaming_engine: true
num_workers: 2
max_num_workers: 32
autoscaling_algorithm: THROUGHPUT_BASED
worker_machine_type: n2-standard-4
disk_size_gb: 100
use_runner_v2: true
experiments:
  - enable_streaming_auto_sharding
```

```bash
# Update a running streaming job without draining state
gcloud dataflow flex-template run orders-stream-v2 --region=us-central1 \
  --template-file-gcs-location=gs://acme-code/templates/orders-v2.json \
  --parameters=inputTopic=projects/acme-data-prod/topics/orders-v1 \
  --max-workers=32 --enable-streaming-engine \
  --update --job-name=orders-stream
```

Worked example: an order pipeline sustains 60,000 messages/s at peak and an order-enrichment `DoFn` processes about 5,000 messages/s per vCPU. On an `n2-standard-4` (4 vCPUs) that is 20,000 messages/s per worker, so 3 workers carry the peak and 4 with headroom. Set `num_workers=1`, `max_num_workers=8`, and alert when system lag exceeds 60 seconds, which is the signal that the per-worker rate was optimistic.

The common mistake is to size for average traffic and then let the lag alert fire daily at peak. The second is to raise `max_num_workers` instead of investigating a hot key, which adds cost without adding throughput because one key still lands on one worker.

In [ ]:
import math

peak_messages_per_second = 60_000
per_vcpu = 5_000
worker_vcpus = 4
headroom = 1.3

workers = math.ceil(peak_messages_per_second / (per_vcpu * worker_vcpus) * headroom)
num_workers_floor = 1
max_workers = 8
assert workers == 4
assert num_workers_floor <= workers <= max_workers
print(f"peak needs {workers} workers; floor {num_workers_floor}, cap {max_workers}")

## Tuning Pub/Sub throughput

Pub/Sub scales horizontally with the number of subscriptions and the number of outstanding messages, not with a partition count you set. A single subscription can deliver a very high rate through streaming pull with many client streams, and Dataflow creates enough reader threads per worker. Ordering keys change that: messages with the same key are delivered in order to one subscriber, so a hot key serializes that key's throughput. Use an ordering key with many distinct values, or accept out-of-order delivery.

The ack deadline is the other throughput control. The default is 10 seconds; a handler that takes longer must either set `--ack-deadline` higher (to 600 seconds) or extend the lease with `modify_ack_deadline`. If the deadline expires before the ack, the message is redelivered and the same work runs twice, which raises the effective rate needed for the same progress. Exactly-once delivery suppresses redelivery after a successful ack, but only within the same subscription.

Two metrics decide whether the system is keeping up: `subscription/oldest_unacked_message_age` is the age of the oldest message still waiting for an ack, and `subscription/num_undelivered_messages` is the backlog size. Growing backlog with flat ack rate means consumers are undersized. A dead-letter topic with `max_delivery_attempts` parks poison messages so they stop consuming the ack budget, but the Pub/Sub service account needs publisher permission on the dead-letter topic.

```python
# Streaming pull with flow control and an explicit ack deadline
from google.cloud import pubsub_v1

subscriber = pubsub_v1.SubscriberClient()
flow = pubsub_v1.types.FlowControl(max_messages=1000, max_bytes=100 * 1024 * 1024)
settings = pubsub_v1.types.StreamingPullRequest(stream_ack_deadline_seconds=60)
assert flow.max_messages == 1000
```

```bash
# Watch the two numbers that matter, then read the dead-letter topic
# pubsub.googleapis.com/subscription/oldest_unacked_message_age
# pubsub.googleapis.com/subscription/num_undelivered_messages
gcloud pubsub subscriptions describe orders-v1-sub \
  --format='value(deadLetterPolicy.maxDeliveryAttempts, ackDeadlineSeconds)'
gcloud pubsub subscriptions pull orders-v1-deadletter --limit=5 --auto-ack
```

Worked example: publish rate is 50,000 messages/s and three subscribers ack 12,000/s each, so the net growth is 14,000 messages/s. The backlog passes one million in 72 seconds, and a 60-second alert delay means the page arrives after the queue is already deep. The fix is either more ack capacity or a lower publish burst, and the measurement that proves it is `num_undelivered_messages` returning to a sawtooth instead of climbing.

The common mistake is to crank `max_messages` on the subscriber without checking the handler's latency. A larger buffer with a slow handler just moves the backlog into client memory and lets ack deadlines expire on messages that were never processed.

In [ ]:
import math

publish_rate = 50_000
subscribers = 3
ack_per_subscriber = 12_000

net_rate = publish_rate - subscribers * ack_per_subscriber
assert net_rate == 14_000
backlog_after_600s = net_rate * 600
seconds_to_one_million = math.ceil(1_000_000 / net_rate)
assert backlog_after_600s == 8_400_000
assert seconds_to_one_million == 72
print(f"net {net_rate}/s, 10 min backlog {backlog_after_600s}, 1M in {seconds_to_one_million}s")

## Controlling Dataproc and Composer cost

Dataproc cost is cluster-minutes times machine price. Three habits dominate. Use ephemeral clusters with `--max-idle` so nothing idles overnight. Put shuffle-heavy secondary workers on Spot VMs, which are discounted but can be preempted, and keep primary workers on standard VMs so the driver and the HDFS name node survive. Use an autoscaling policy that adds secondary workers from YARN pending memory and removes them from idle time, and prefer Dataproc Serverless for batch jobs that do not need a persistent cluster.

Composer cost is environment-hours plus worker count, billed whether or not a DAG runs. Environment size sets the minimum worker count and scheduler capacity. To reduce cost: right-size the environment, avoid long-blocking sensors in favour of deferrable operators that release the slot, keep DAG files small so parse time stays low, and use `max_active_runs` and pools to prevent a backfill from scaling the environment. A small environment plus more DAG discipline is usually cheaper than a large environment.

```bash
# Ephemeral cluster with a Spot shuffle tier and an autoscaling policy
gcloud dataproc clusters create etl-tmp --region=us-central1 \
  --num-workers=4 --worker-machine-type=n2-standard-8 \
  --num-secondary-workers=20 --secondary-worker-type=spot \
  --autoscaling-policy=orders-etl --max-idle=30m --enable-component-gateway
gcloud dataproc jobs submit pyspark gs://acme-code/orders_etl.py \
  --cluster=etl-tmp --region=us-central1 --properties=spark.sql.shuffle.partitions=400
gcloud dataproc clusters delete etl-tmp --region=us-central1 --quiet
```

```yaml
# Autoscaling policy from YARN metrics: scale secondary workers between bounds
autoscalingPolicy:
  id: orders-etl
  basicAlgorithm:
    yarnConfig:
      scaleUpFactor: 1.0
      scaleDownFactor: 1.0
      scaleUpMinWorkerFraction: 0.1
      scaleDownMinWorkerFraction: 0.1
      gracefulDecommissionTimeout: 300s
  workerConfig:
    minInstances: 4
    maxInstances: 32
    weight: 1
  secondaryWorkerConfig:
    minInstances: 0
    maxInstances: 40
    weight: 1
```

```sql
-- Attribute Dataproc and Composer spend by label from the billing export
SELECT service.description AS service, IFNULL(labels.value, 'unlabeled') AS team,
       ROUND(SUM(cost), 2) AS usd, COUNT(*) AS line_items
FROM `acme-billing.billing_export.gcp_billing_export_v1_0123AB` AS b
LEFT JOIN UNNEST(b.labels) AS labels
WHERE invoice.month = '202609'
  AND service.description IN ('Cloud Dataproc', 'Cloud Composer')
GROUP BY service, team ORDER BY usd DESC;
```

Worked example: a PySpark job runs 40 minutes with 4 primary workers and 20 Spot secondary workers. On-demand `n2-standard-8` is the reference price for primaries; Spot secondaries cost roughly a third of that. Running the same job nightly on a persistent 24-worker cluster costs about 23 hours of idle cluster time per day, which is the number that usually ends the debate.

The common mistake is to keep one cluster for many teams because creation takes a few minutes. The second is to autoscale primary workers on Spot instances, which can preempt the driver mid-job. Scale the secondary tier and protect the primary tier.

In [ ]:
import math

# Ephemeral cluster cost model; prices are inputs from the current rate card
primary_hourly = 0.40
spot_hourly = 0.13
job_minutes = 40
idle_hours_per_day = 23

hourly = 4 * primary_hourly + 20 * spot_hourly
ephemeral = hourly * (job_minutes / 60)
persistent_idle = hourly * idle_hours_per_day
assert round(ephemeral, 2) == 2.80
assert round(persistent_idle, 2) == 96.60
assert persistent_idle > ephemeral * 20
print(f"ephemeral run ${ephemeral:.2f}; idle cluster ${persistent_idle:.2f}/day")

## Things to remember: key numbers and limits

Numbers that come up in design reviews. Prices are illustrative us list prices at the time of writing; limits are stable.

| Topic | Number / fact | Why it matters |
|---|---|---|
| BigQuery pricing | on-demand about 6.25 USD per TiB scanned, 10 MiB minimum per query | a missing partition filter is a cost event |
| GCS classes | Nearline minimum 30 days, Coldline 90, Archive 365 | deleting early incurs the remaining duration |
| GCS objects | immutable; updates are rewrites | plan layout for append-by-partition, not in-place edits |
| File size target | roughly 128-512 MB files on GCS | engines pay per file in listing and planning |
| Dataflow | one vCPU sustains roughly 5,000 1-KB messages per second | worker count comes from throughput and per-vCPU rate |
| Pub/Sub | delivery needs a subscription; a subscription is a durable cursor, not a queue entry | messages without a subscription are dropped; backlog only exists per subscription |
| Pub/Sub ordering | guaranteed per ordering key with ordering enabled | the key, not the topic, is the unit of order |
| Composer | ~0.35 USD/hour environment fee plus workers, billed 24/7 while the environment exists | it is the one service here that costs money while idle |
| Spanner | 1,000 processing units = 1 node | capacity is bought in units; reads and storage scale separately |
| BigQuery hierarchy | project -> dataset -> table; IAM at any level | dataset-level grants are the usual least-privilege unit |

## Things to remember: semantics to memorize

* **Pub/Sub is a log with cursors.** Each subscription gets its own cursor and its own backlog — adding a subscriber never competes with existing ones, and seek replays from a timestamp.
* **BigQuery bills bytes scanned, not rows returned.** Partition filters, clustering, and selecting only needed columns are the cost controls.
* **Beam code is runner-independent.** The same pipeline runs on DirectRunner for tests and on Dataflow in production — test locally, deploy unchanged.
* **An ephemeral Dataproc cluster is the default.** Create per job, delete on completion; a long-lived shared cluster pays for idle time and lock-in.
* **IAM is allow-only and additive.** You cannot subtract a parent grant with a narrower policy; Conditions and deny policies are the exceptions to know.
* **VPC Service Controls is the exfiltration control.** IAM says who can call an API; the perimeter says the data cannot leave even for an owner without an access level.
* **Every byte is encrypted at rest by default.** CMEK puts the key under your control for rotation and revocation; CSEK means you hold the key entirely.

## Real-world use cases on GCP

### 1. Retail demand forecasting on BigQuery and Vertex AI

Business context: a 900-store retailer forecasts SKU demand weekly across 250,000 SKU-store combinations; planners need a forecast by Monday 06:00 and the pipeline must handle Black Friday spikes. Volume: 1.2B sales rows/year, 380 GB in BigQuery, 40M rows/day during peak.

```text
POS + e-commerce -> Pub/Sub / batch to GCS -> Dataflow -> BigQuery curated
   -> dbt/BigQuery SQL features -> Vertex AI AutoML forecasting
   -> predictions back to BigQuery -> Looker/Tableau demand dashboard
```

```sql
-- Feature mart partitioned by week and clustered by store/SKU for the model export
CREATE OR REPLACE TABLE ml.demand_features
PARTITION BY week_start
CLUSTER BY store_id, sku_id AS
SELECT
  DATE_TRUNC(sale_date, WEEK) AS week_start,
  store_id, sku_id,
  SUM(units)                        AS units_7d,
  AVG(units)                        AS units_avg_28d,
  SUM(promo_flag)                   AS promo_days,
  MAX(temperature_c)                AS max_temp,
  EXTRACT(ISOWEEK FROM sale_date)   AS iso_week
FROM curated.sales
WHERE sale_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 730 DAY)
GROUP BY 1, 2, 3, 7;
```

Why this design: BigQuery holds the full history cheaply, partitioning by week keeps both training exports and planner queries to a bounded scan, and Vertex AI forecasting consumes the feature table directly, so there is one governed definition of demand instead of duplicated logic in the BI layer.

### 2. Fintech ledger with Datastream CDC and near-real-time balances

Business context: a payments company needs a near-current merchant balance and a fully auditable ledger for regulators. Volume: 25M transactions/day (~2,000/s peak), 5 KB/event, 7-year retention.

```text
Cloud SQL (Postgres) -> Datastream CDC -> GCS landing + Pub/Sub stream
   -> Dataflow: dedupe by (txn_id, lsn), window 1 min -> BigQuery ledger_current
   -> nightly reconciliation job -> BigQuery ledger_archive (7-year, partitioned)
   -> Compliance dashboard: live connection with row-level security by region
```

```sql
-- Idempotent near-real-time merge keyed on the CDC log position
MERGE curated.ledger_current AS t
USING (
  SELECT txn_id, account_id, amount, currency, txn_ts, lsn
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY txn_id ORDER BY lsn DESC) AS rn
    FROM staging.ledger_changes
    WHERE DATE(ingest_ts) = CURRENT_DATE()
  ) WHERE rn = 1
) AS s
ON t.txn_id = s.txn_id AND s.lsn >= t.lsn
WHEN MATCHED AND s.lsn > t.lsn THEN UPDATE SET
  t.amount = s.amount, t.lsn = s.lsn, t.txn_ts = s.txn_ts
WHEN NOT MATCHED THEN INSERT (txn_id, account_id, amount, currency, txn_ts, lsn)
VALUES (s.txn_id, s.account_id, s.amount, s.currency, s.txn_ts, s.lsn);
```

Why this design: Datastream captures deletes and updates that timestamp polling would miss, the `lsn` version makes replays safe, and separating the current-state table (small, hot) from the immutable archive (large, cold) keeps the compliance query affordable for seven years.

### 3. Streaming IoT telemetry for fleet operations

Business context: a logistics operator monitors 18,000 vehicles; operations needs alerts within 30 seconds for engine faults, and data science needs historical telemetry for predictive maintenance. Volume: 18,000 devices x 1 message/10s = 1,800 msg/s, ~4 TB/month raw.

```text
vehicle gateway -> MQTT broker on GKE -> Pub/Sub (key = vehicle_id)
   -> Dataflow streaming: 30s windows, threshold + moving-average anomaly
        -> BigQuery alerts_30s (ops dashboard, live)
        -> GCS raw Parquet (archive)
   -> hourly batch (Dataproc) -> BigQuery telemetry_1m (features + training)
```

```python
# Streaming anomaly detection with per-key state via a sliding window
import json

import apache_beam as beam
from apache_beam.transforms.window import SlidingWindows
from apache_beam.transforms.trigger import AfterWatermark, AfterProcessingTime
from apache_beam.transforms.combiners import MeanCombineFn

FAULT_TEMP = 105.0

with beam.Pipeline() as p:
    (
        p
        | beam.io.ReadFromPubSub(topic="projects/acme/topics/vehicle-telemetry")
        | beam.Map(lambda b: json.loads(b.decode("utf-8")))
        | beam.Map(lambda e: (e["vehicle_id"], e["engine_temp_c"]))
        | beam.WindowInto(SlidingWindows(60, 10),
                          trigger=AfterWatermark(late=AfterProcessingTime(30)),
                          accumulation_mode=AccumulationMode.DISCARDING)
        | beam.CombinePerKey(MeanCombineFn())
        | beam.Filter(lambda kv: kv[1] >= FAULT_TEMP)
        | beam.io.WriteToBigQuery("acme:gold.engine_faults",
                                  schema="vehicle_id:STRING,mean_temp:FLOAT",
                                  write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND)
    )
```

Why this design: a sliding 60-second window smooths sensor noise so a single spike does not page a mechanic, updating the same code with `--streaming` keeps one pipeline for cloud and on-prem testing, and the heavy feature engineering runs hourly on Parquet where scan cost is low.

### 4. Multi-tenant SaaS product analytics with VPC Service Controls

Business context: a B2B SaaS product serves 1,100 tenants; each tenant's admins see only their usage data, and the platform team needs cross-tenant health metrics. Volume: 2.5B events/month, 60 TB in the lakehouse, 400 concurrent dashboard users.

```text
tenant SDKs -> Pub/Sub -> Dataflow -> BigQuery raw.events (tenant_id in key)
   -> scheduled SQL -> curated.tenant_usage_daily
   -> Looker/embedded dashboards with row-level security on tenant_id
   -> VPC Service Controls perimeter around the project for exfiltration control
```

```sql
CREATE ROW ACCESS POLICY tenant_isolation
ON curated.tenant_usage_daily
GRANT TO ('group:tenant-admins@acme.com')
FILTER USING (tenant_id = SESSION_USER());

-- Cross-tenant health for the platform team uses a separate authorized view
CREATE OR REPLACE VIEW platform.tenant_health AS
SELECT tenant_id, usage_date, active_users, events,
       active_users / NULLIF(licensed_seats, 0) AS seat_utilization
FROM curated.tenant_usage_daily;
```

Why this design: `tenant_id` in the partition and cluster keys keeps every tenant query to its own slice, row-level security means the dashboard never has to trust the application to filter, and the VPC Service Controls perimeter makes bulk exfiltration of the tenant tables detectable and blocked.

### 5. Marketing attribution pipeline with Dataproc Serverless

Business context: a D2C brand blends six ad platforms, web clickstream, and CRM data to compute CAC and ROAS daily; the existing Spark jobs came from an on-prem Hadoop cluster. Volume: 90M impressions/day, 14M sessions/day, 220K orders/day, 12 TB/month in GCS.

```text
ad APIs -> GCS landing (Avro) ---\
web/app clickstream -> Pub/Sub -> GCS --\
CRM CDC -> Datastream -> GCS ------------> Dataproc Serverless (PySpark)
   -> BigQuery session_attribution + channel_cac_daily
   -> Tableau marketing dashboard
```

The attribution job is existing Spark code (last-touch within 30 days, then aggregate by day and channel), submitted to Dataproc Serverless unchanged. The same logic in SQL, runnable locally with DuckDB:

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE sessions AS SELECT * FROM (VALUES
        ('u1', 'search', TIMESTAMP '2026-09-01 10:00'), ('u1', 'email', TIMESTAMP '2026-09-20 09:00'),
        ('u2', 'social', TIMESTAMP '2026-08-01 12:00'), ('u2', 'search', TIMESTAMP '2026-09-25 08:00'),
        ('u3', 'social', TIMESTAMP '2026-07-01 12:00')
    ) t(user_id, channel, session_ts);
    CREATE TABLE conversions AS SELECT * FROM (VALUES
        ('c1', 'u1', TIMESTAMP '2026-09-21 18:00', 120.0), ('c2', 'u2', TIMESTAMP '2026-09-26 11:00', 80.0),
        ('c3', 'u3', TIMESTAMP '2026-09-26 12:00', 50.0)  -- no session in the 30-day lookback
    ) t(conversion_id, user_id, conversion_ts, revenue);
""")
attribution = con.execute("""
    WITH last_touch AS (
        SELECT c.conversion_id, c.conversion_ts, c.revenue, s.channel
        FROM conversions c
        JOIN sessions s
          ON s.user_id = c.user_id
         AND s.session_ts BETWEEN c.conversion_ts - INTERVAL 30 DAY AND c.conversion_ts
        QUALIFY row_number() OVER (PARTITION BY c.conversion_id ORDER BY s.session_ts DESC) = 1
    )
    SELECT CAST(conversion_ts AS DATE) AS ds, channel,
           count(DISTINCT conversion_id) AS conversions, sum(revenue) AS revenue
    FROM last_touch GROUP BY ALL ORDER BY ds, channel
""").fetchall()
for row in attribution:
    print(row)
assert [(r[1], r[3]) for r in attribution] == [("email", 120.0), ("search", 80.0)]

```bash
# Submit without a cluster; Dataproc Serverless sizes the batch automatically
gcloud dataproc batches submit pyspark gs://acme-code/attribution.py \
  --region=us-central1 --deps-bucket=gs://acme-code/deps \
  --service-account=dataproc-runner@acme-data-prod.iam.gserviceaccount.com \
  --properties=spark.executor.cores=4,spark.executor.instances=20
```

Why this design: the team keeps its Spark code and libraries (no Beam rewrite), Dataproc Serverless removes the idle-cluster cost of the old on-prem model, and BigQuery becomes the serving layer for the marketing dashboard so attribution logic lives in one place rather than in every workbook.

## GCP exercises

These use only the standard library; each models a real GCP sizing or cost decision. Run each cell and read the asserts.

### Try it 1. Size Dataflow workers from throughput

Task: a streaming job reads 250,000 messages/s, each about 1 KB. One vCPU sustains about 5,000 messages/s for this enrichment, and the worker type has 4 vCPUs. Size the pool with 30 percent headroom and cap it at 64 workers.

Hint: workers needed equals the ceiling of the input rate divided by the per-worker rate, multiplied by headroom, then clamped to the cap.

Expected result: 13 workers before headroom, 17 after headroom, so 17 is the answer.

In [ ]:
import math

messages_per_second = 250_000
per_vcpu = 5_000
vcpus_per_worker = 4
headroom = 1.3
worker_cap = 64

bare = math.ceil(messages_per_second / (per_vcpu * vcpus_per_worker))
with_headroom = math.ceil(bare * headroom)
workers = min(with_headroom, worker_cap)

assert bare == 13
assert with_headroom == 17
assert workers == 17
print(f"bare={bare} with_headroom={with_headroom} workers={workers}")

### Try it 2. Compute Pub/Sub backlog growth from publish and ack rates

Task: publishers send 50,000 messages/s. Three subscribers each ack 12,000 messages/s. Compute the net growth rate, the backlog after 600 seconds, and the whole seconds until the backlog reaches one million messages.

Hint: net rate is publish minus the summed ack rate. Backlog is rate times time. Time to a threshold is the ceiling of threshold divided by the net rate.

Expected result: net 14,000 messages/s, 8,400,000 messages after 600 s, one million reached in about 72 seconds.

In [ ]:
import math

publish_rate = 50_000
subscribers = 3
ack_per_subscriber = 12_000
window_seconds = 600
threshold = 1_000_000

net_rate = publish_rate - subscribers * ack_per_subscriber
backlog = net_rate * window_seconds
seconds_to_threshold = math.ceil(threshold / net_rate)

assert net_rate == 14_000
assert backlog == 8_400_000
assert seconds_to_threshold == 72
print(f"net={net_rate}/s backlog_600s={backlog} threshold_in={seconds_to_threshold}s")

### Try it 3. Choose a GCS storage class by access pattern

Task: four prefixes each hold 1,000 GB. Last access was 7, 45, 200, and 500 days ago. Classify each as STANDARD, NEARLINE, COLDLINE, or ARCHIVE using the access-age rule, then total the illustrative monthly storage cost at 0.020, 0.010, 0.004, and 0.0012 dollars per GB-month.

Hint: within 30 days is STANDARD, within 90 is NEARLINE, within 365 is COLDLINE, older is ARCHIVE. The rates are illustrative: use your current rate card.

Expected result: STANDARD, NEARLINE, COLDLINE, ARCHIVE and a total of 35.20 dollars per month.

In [ ]:
gb = 1000.0
rates = {"STANDARD": 0.020, "NEARLINE": 0.010, "COLDLINE": 0.004, "ARCHIVE": 0.0012}
last_access_days = [7, 45, 200, 500]

def classify(days):
    if days <= 30:
        return "STANDARD"
    if days <= 90:
        return "NEARLINE"
    if days <= 365:
        return "COLDLINE"
    return "ARCHIVE"

classes = [classify(d) for d in last_access_days]
total = sum(gb * rates[c] for c in classes)

assert classes == ["STANDARD", "NEARLINE", "COLDLINE", "ARCHIVE"]
assert round(total, 2) == 35.20
print(classes, f"monthly=${total:.2f}")

### Try it 4. Estimate BigQuery cost from bytes

Task: a dashboard query scans 4.2 TiB and an ad hoc query scans 6 MiB. On-demand pricing is 6.25 dollars per TiB, and BigQuery bills a minimum of 10 MiB per query. Compute both costs.

Hint: billed bytes are the maximum of scanned bytes and the 10 MiB minimum, and cost is billed bytes divided by bytes in a TiB times the rate.

Expected result: about 26.25 dollars for the dashboard query and about 0.0000596 dollars for the ad hoc query.

In [ ]:
tib = 1024 ** 4
mib = 1024 ** 2
rate_per_tib = 6.25
minimum_billed = 10 * mib

big_scanned = 4.2 * tib
small_scanned = 6 * mib

cost_big = big_scanned / tib * rate_per_tib
billed_small = max(small_scanned, minimum_billed)
cost_small = billed_small / tib * rate_per_tib

assert round(cost_big, 2) == 26.25
assert billed_small == minimum_billed
assert abs(cost_small - 6.25 * 10 / 1024 / 1024) < 1e-12
print(f"dashboard=${cost_big:.2f} ad_hoc=${cost_small:.8f}")

### Try it 5. Size Dataproc autoscaling from job stages

Task: a Spark job has three stages with (tasks, tasks per worker) of (400, 8), (120, 4), and (64, 8). Compute the workers each stage needs, take the maximum as the peak, set the autoscaling maximum to 150 percent of peak rounded up, and set the minimum to the ceiling of peak divided by 4 with a floor of 2.

Hint: workers per stage is the ceiling of tasks divided by tasks per worker. Peak is the max across stages. Min and max are derived from peak.

Expected result: peak 50 workers, min 13, max 75.

In [ ]:
import math

stages = [("scan", 400, 8), ("shuffle", 120, 4), ("write", 64, 8)]

per_stage = {name: math.ceil(tasks / per_worker) for name, tasks, per_worker in stages}
peak = max(per_stage.values())
min_workers = max(2, math.ceil(peak / 4))
max_workers = math.ceil(peak * 1.5)

assert per_stage == {"scan": 50, "shuffle": 30, "write": 8}
assert peak == 50
assert min_workers == 13
assert max_workers == 75
print(per_stage, f"min={min_workers} max={max_workers}")

### Try it 6. Compute Composer worker cost

Task: a Composer environment bills an environment fee of 0.35 dollars per hour plus 3 workers at 0.10 dollars per hour each. Compute the cost for a 730-hour month, then compute what a 5-worker environment with the same fee would cost and the monthly difference.

Hint: cost equals hours times (fee plus workers times worker rate). Prices here are illustrative inputs; substitute your current rate card.

Expected result: 474.50 dollars for 3 workers, 620.50 dollars for 5 workers, a difference of 146.00 dollars.

In [ ]:
hours = 730
environment_fee = 0.35
worker_rate = 0.10

small = hours * (environment_fee + 3 * worker_rate)
large = hours * (environment_fee + 5 * worker_rate)
delta = large - small

assert round(small, 2) == 474.50
assert round(large, 2) == 620.50
assert round(delta, 2) == 146.00
print(f"3 workers ${small:.2f}, 5 workers ${large:.2f}, delta ${delta:.2f}/month")

### Try it 7. Plan a GCS lifecycle policy

Task: build a lifecycle rule list for prefix orders/ that transitions to NEARLINE at 30 days, COLDLINE at 120 days, ARCHIVE at 365 days, and deletes at 2555 days. Assert the ages strictly increase and that each transition respects the minimum storage duration of the destination class: NEARLINE 30, COLDLINE 90, ARCHIVE 365 days.

Hint: minimum storage duration applies to the class the object leaves, so each transition age must be at least the minimum of the class being left.

Expected result: four rules, the first transition at 30 days, and every transition age greater than the minimum duration of the previous class.

In [ ]:
rules = [
    {"action": "SetStorageClass:NEARLINE", "age": 30},
    {"action": "SetStorageClass:COLDLINE", "age": 120},
    {"action": "SetStorageClass:ARCHIVE", "age": 365},
    {"action": "Delete", "age": 2555},
]
min_duration = {"NEARLINE": 30, "COLDLINE": 90, "ARCHIVE": 365}
ages = [r["age"] for r in rules]

assert len(rules) == 4
assert ages == sorted(ages) and len(set(ages)) == 4
assert ages[0] == 30
previous_class = "NEARLINE"
for rule, age in zip(rules, ages):
    assert age >= min_duration[previous_class]
    if rule["action"].startswith("SetStorageClass:"):
        previous_class = rule["action"].split(":")[1]
assert rules[-1]["action"] == "Delete"
print("lifecycle ages:", ages)